# Two-terminal QH conductance with Kwant — cropped region + recomputed chemical potential

This notebook is identical in workflow to `qh_kwant_transport_two_terminal_JOBLIB.ipynb`, with one geometry optimization: the full Hartree potential is first reshaped under the source/drain mouths exactly as in the original v4 workflow, and **only afterwards** the reshaped outer regions are discarded.

For the left/right current-lead depth `source_drain_depth_nm`, the transport system retains only

```text
xmin + source_drain_depth_nm < x < xmax - source_drain_depth_nm
```

(equivalently, for a symmetric grid `-d + source_drain_depth_nm < x < d - source_drain_depth_nm`).

The source and drain are then attached to the new left/right boundaries of this reduced system. All other physics parameters, energy selection, magnetic-field treatment, JOBLIB sweep logic, observables, and saved output keys are unchanged.

Lead ordering used throughout:

```text
0 = source, left full-width lead
1 = drain, right full-width lead
```


**Added in this version:** recomputation of the chemical potential from the cropped-region particle-number constraint, density diagnostics, and a direct conductance comparison using the original and recomputed chemical potentials.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from scipy.optimize import brentq
import numpy as np
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
import multiprocessing

from matplotlib.animation import FuncAnimation
from IPython.display import HTML


#from qh_kwant_transport_mu_metadata_two_terminal import *
#from qh_kwant_transport_mu_metadata_two_terminal_v2 import *
#from qh_kwant_transport_mu_metadata_two_terminal_v3 import *
from qh_kwant_transport_mu_metadata_two_terminal_v4 import *

from modules.simulation_io import (
    prepare_simulation,
    print_simulation_info,
    params_to_json_dict,
    json_safe_value,
)
from modules.grid_kernel_confinement_io import make_kernel_and_background_2d
from modules.hartree_io import (
    solve_self_consistent_2d,
    average_density_2d,
    prepare_landau_context,
    density_from_potential_2d,
)


## 1. Load the Hartree sweep

Set `SWEEP_DIR` to the directory containing subfolders with `metadata.json` and `data.npz`.
The potential key is usually `"U"`.

In [ ]:
SWEEP_DIR = Path("sweep_fold")
POTENTIAL_KEY = "U"

seq = load_effective_potential_sequence(
    SWEEP_DIR,
    potential_key=POTENTIAL_KEY,
    ground_each_frame=False,
)

print(seq.U_stack_meV.shape, "(nB, ny, nx)")
print("B range:", seq.B_values_T[0], "->", seq.B_values_T[-1], "T")
print("x range:", seq.x_nm.min(), "->", seq.x_nm.max(), "nm")
print("y range:", seq.y_nm.min(), "->", seq.y_nm.max(), "nm")
print("grid spacing a =", seq.a_nm, "nm")

## 2. Define a two-terminal Hall bar

The source/drain potential reshaping is performed on the **full** Hartree grid first. A thin wrapper around the v4 preparation routine then removes the two reshaped outer sections before the Kwant builder is called. This keeps the original v4 logic intact while substantially reducing the number of lattice sites.


In [ ]:
# Choose one representative frame for visualization/building.
FRAME_INDEX = 1  #min(101, len(seq.B_values_T) - 1)

# Lead-mouth parameters.
source_drain_depth_nm = 700.0     # reshaping depth for left/right current leads

lead_potential_meV = 0.0
smooth_nm = 1e1

lead_specs_2 = (
    LeadSpec(
        "source", "left",
        depth_nm=source_drain_depth_nm,
        lead_potential_meV=lead_potential_meV,
        smooth_nm=smooth_nm,
        smooth_profile="constant_inner",
    ),
    LeadSpec(
        "drain", "right",
        depth_nm=source_drain_depth_nm,
        lead_potential_meV=lead_potential_meV,
        smooth_nm=smooth_nm,
        smooth_profile="constant_inner",
    ),
)

cfg2 = DeviceConfig(
    m_eff=0.067,
    g_factor=-0.0,
    norbs=1,
    disorder=DisorderConfig(amplitude_meV=1e-6, correlation_nm=15.0, seed=12345),
    lead_specs=lead_specs_2,
    reshape_under_leads=True,
    lead_B_mode="zero",
)


# -------------------------------------------------------------------------
# Geometry optimization:
# 1) run the ORIGINAL v4 prepare_frame on the full Hartree grid, so the
#    source/drain mouths are reshaped exactly as before;
# 2) only afterwards discard the reshaped left/right sections;
# 3) make run_one_frame use this same cropped preparation also inside
#    JOBLIB worker processes.
# -------------------------------------------------------------------------

_prepare_frame_full = prepare_frame
_run_one_frame_full = run_one_frame


def crop_prepared_frame_x(prep, crop_depth_nm):
    """
    Return a copy of a v4 PreparedFrame restricted to the inner x interval

        xmin + crop_depth_nm < x < xmax - crop_depth_nm.

    The crop is applied to x_nm and to every prepared ndarray whose final
    spatial dimensions are (ny, nx), plus any 1D x-dependent array.

    This function deliberately acts AFTER the original v4 prepare_frame,
    hence U_for_transport_meV has already undergone the original lead-mouth
    reshaping before the outer regions are thrown away.
    """
    import copy
    import dataclasses

    x = np.asarray(prep.x_nm)
    y = np.asarray(prep.y_nm)
    nx = len(x)
    ny = len(y)

    xmin = float(x.min())
    xmax = float(x.max())

    x_left = xmin + float(crop_depth_nm)
    x_right = xmax - float(crop_depth_nm)

    keep_x = (x > x_left) & (x < x_right)

    if np.count_nonzero(keep_x) < 2:
        raise ValueError(
            "source_drain_depth_nm removes the whole transport region: "
            f"full x range = [{xmin}, {xmax}] nm, "
            f"requested retained interval = ({x_left}, {x_right}) nm."
        )

    def crop_value(name, value):
        # Coordinate arrays must be treated explicitly.
        if name == "x_nm":
            return np.asarray(value)[keep_x]

        if name == "y_nm":
            return np.asarray(value)   # NEVER crop y

        if not isinstance(value, np.ndarray):
            return value

        # Spatial maps: crop ONLY their x axis.
        if value.ndim >= 2 and value.shape[-2:] == (ny, nx):
            sl = [slice(None)] * value.ndim
            sl[-1] = keep_x
            return value[tuple(sl)]

        # 1D arrays known to live on x.
        # Only use this generic rule after excluding y_nm explicitly.
        if value.ndim == 1 and value.shape[0] == nx:
            return value[keep_x]

        return value

    if dataclasses.is_dataclass(prep):
        updates = {}
        for f in dataclasses.fields(prep):
            old = getattr(prep, f.name)
            new = crop_value(f.name, old)
            if new is not old:
                updates[f.name] = new
        cropped = dataclasses.replace(prep, **updates)
    else:
        # Fallback for a non-dataclass PreparedFrame implementation.
        cropped = copy.copy(prep)
        for name, old in vars(prep).items():
            new = crop_value(name, old)
            if new is not old:
                setattr(cropped, name, new)

    return cropped


def prepare_frame(seq, index, cfg):
    """Original v4 frame preparation followed by the inner-x crop."""
    prep_full = _prepare_frame_full(seq, index, cfg)
    return crop_prepared_frame_x(prep_full, source_drain_depth_nm)


def run_one_frame(*args, **kwargs):
    """
    Call the original v4 run_one_frame, but temporarily replace the module's
    prepare_frame by the cropped wrapper above.

    Doing the patch inside this function is intentional: JOBLIB/loky workers
    import the v4 module in separate processes, so each worker must activate
    the cropped preparation locally before running a frame.
    """
    module_globals = _run_one_frame_full.__globals__
    previous_prepare_frame = module_globals.get("prepare_frame")
    module_globals["prepare_frame"] = prepare_frame
    try:
        return _run_one_frame_full(*args, **kwargs)
    finally:
        module_globals["prepare_frame"] = previous_prepare_frame


# Prepare one frame through the NEW cropped path.
prep2 = prepare_frame(seq, FRAME_INDEX, cfg2)

# Also prepare the corresponding full frame only for diagnostics.
prep2_full = _prepare_frame_full(seq, FRAME_INDEX, cfg2)

print("lead order:")
for i, spec in enumerate(cfg2.lead_specs):
    print(i, spec)
print("B =", prep2.B_T, "T")
print("t =", prep2.t_meV, "meV")
print("alpha =", prep2.alpha)
print("lB =", prep2.lB_nm, "nm")
print("full grid shape   =", prep2_full.U_for_transport_meV.shape)
print("cropped grid shape=", prep2.U_for_transport_meV.shape)
print(
    "x retained:",
    float(prep2.x_nm.min()), "->", float(prep2.x_nm.max()), "nm",
)
print(
    "site-count reduction factor ≈",
    prep2_full.U_for_transport_meV.size / prep2.U_for_transport_meV.size,
)


## 2b. Recompute the chemical potential on the cropped transport potential

This section is the main addition to the original notebook.

For every magnetic-field frame it prepares the **same cropped potential passed to Kwant**, evaluates the Hartree/Thomas-Fermi density on that fixed potential, and solves for a new chemical potential \(\mu_{\rm crop}(B)\) that restores the requested average electron density.

The default target is the **saved Hartree electron density averaged over exactly the region that survives the x-crop**. If no saved density array is found, the code falls back to `metadata["params"]["n_target"]`.

This is intentionally **not** a new self-consistent Hartree calculation: \(U\) is kept fixed. The purpose is to isolate whether cropping / transport preprocessing makes the original Hartree chemical potential inconsistent with the density of the actual Kwant potential.


In [ ]:
@dataclass
class DensityParams2D:
    Lx: float = 3000.0
    Ly: float = 3000.0
    Nx: int = 301
    Ny: int = 301

    eps_r: float = 13.0
    n_target: float = 0.7e-3

    B: float = 1.0
    g_spin: float = 0.0
    n_LL: int = 8
    gamma_rel: float = 0.015
    use_B_scaling: bool = True

    @property
    def hbar_omega_c(self):
        return 1.76 * self.B

    @property
    def lB(self):
        return 25.658 / np.sqrt(self.B)

    @property
    def Gamma(self):
        if self.use_B_scaling:
            return self.gamma_rel * self.hbar_omega_c * np.sqrt(10.0 / self.B)
        return self.gamma_rel * self.hbar_omega_c


# "saved_cropped_density" preserves the average density that the retained
# region had in the original Hartree solution.
# "metadata_n_target" instead enforces the global metadata n_target.
TARGET_DENSITY_MODE = "saved_cropped_density"

SAVED_DENSITY_KEYS = (
    "n",
    "n_e",
    "ne",
    "density",
    "electron_density",
    "electron_density_nm2",
)

MU_BRACKET_LL_MARGIN = 4.0


In [ ]:
def _nested_get(d, path, default=None):
    cur = d
    for key in path.split("."):
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def density_params_for_frame(index, prep=None):
    frame = seq.frame(index)
    meta_params = frame.metadata.get("params", {}) if isinstance(frame.metadata, dict) else {}

    pden = DensityParams2D()

    for name in (
        "eps_r",
        "n_target",
        "g_spin",
        "n_LL",
        "gamma_rel",
        "use_B_scaling",
    ):
        if name in meta_params:
            setattr(pden, name, meta_params[name])

    pden.B = float(frame.B_T)

    if prep is not None:
        pden.Nx = int(len(prep.x_nm))
        pden.Ny = int(len(prep.y_nm))
        if len(prep.x_nm) > 1:
            pden.Lx = float(prep.x_nm.max() - prep.x_nm.min())
        if len(prep.y_nm) > 1:
            pden.Ly = float(prep.y_nm.max() - prep.y_nm.min())

    return pden


def _full_x_crop_mask():
    x = np.asarray(seq.x_nm)
    xmin = float(x.min())
    xmax = float(x.max())
    x_left = xmin + float(source_drain_depth_nm)
    x_right = xmax - float(source_drain_depth_nm)
    return (x > x_left) & (x < x_right)


def load_saved_cropped_density(index):
    frame = seq.frame(index)
    if frame.run_dir is None:
        return None, None, None

    data_file = Path(frame.run_dir) / "data.npz"
    if not data_file.exists():
        return None, None, None

    keep_x = _full_x_crop_mask()

    with np.load(data_file, allow_pickle=False) as z:
        density_key = next((k for k in SAVED_DENSITY_KEYS if k in z.files), None)
        if density_key is None:
            return None, None, None

        n_full = np.asarray(z[density_key], dtype=float)
        expected_shape = seq.U_stack_meV[index].shape

        if n_full.shape != expected_shape:
            print(
                f"Warning: saved density {density_key!r} has shape {n_full.shape}, "
                f"expected {expected_shape}; falling back to metadata n_target."
            )
            return None, None, None

        n_crop = n_full[:, keep_x]

        if "mask" in z.files:
            mask_full = np.asarray(z["mask"], dtype=bool)
            if mask_full.shape == n_full.shape:
                mask_crop = mask_full[:, keep_x]
            else:
                mask_crop = np.isfinite(n_crop)
        else:
            mask_crop = np.isfinite(n_crop)

    return n_crop, mask_crop, density_key


def mean_density(n_xy, mask=None):
    n_xy = np.asarray(n_xy, dtype=float)

    if mask is None:
        mask = np.isfinite(n_xy)
    else:
        mask = np.asarray(mask, dtype=bool) & np.isfinite(n_xy)

    if not np.any(mask):
        raise ValueError("Density averaging mask contains no valid sites.")

    return float(np.mean(n_xy[mask]))


def density_on_transport_potential(U_meV, mu_meV, pden, mask=None, ll=None):
    if ll is None:
        ll = prepare_landau_context(pden)

    if mask is None:
        mask = np.isfinite(U_meV)

    n_xy = density_from_potential_2d(
        np.asarray(U_meV, dtype=float),
        float(mu_meV),
        pden,
        mask=mask,
        ll=ll,
    )

    return np.asarray(n_xy, dtype=float)


def target_density_for_frame(index):
    frame = seq.frame(index)

    n_saved_crop, saved_mask, density_key = load_saved_cropped_density(index)
    saved_mean = np.nan

    if n_saved_crop is not None:
        saved_mean = mean_density(n_saved_crop, saved_mask)

    if TARGET_DENSITY_MODE == "saved_cropped_density" and np.isfinite(saved_mean):
        return saved_mean, f"saved {density_key} cropped mean", saved_mean

    n_target = _nested_get(frame.metadata, "params.n_target", None)

    if n_target is None:
        raise KeyError(
            "No usable saved density was found and "
            "metadata['params']['n_target'] is missing."
        )

    return float(n_target), "metadata params.n_target", saved_mean


def recompute_mu_for_frame(index, verbose=False):
    frame = seq.frame(index)
    prep = prepare_frame(seq, index, cfg2)

    Utr = np.asarray(prep.U_for_transport_meV, dtype=float)
    mask = np.isfinite(Utr)

    pden = density_params_for_frame(index, prep=prep)
    ll = prepare_landau_context(pden)

    mu_old = float(mu_selector(frame))
    n_target, target_source, n_saved_crop = target_density_for_frame(index)

    def nbar(mu):
        n_xy = density_on_transport_potential(
            Utr,
            mu,
            pden,
            mask=mask,
            ll=ll,
        )
        return mean_density(n_xy, mask)

    n_old = nbar(mu_old)

    hwc = float(pden.hbar_omega_c)

    lo = float(np.nanmin(Utr) - MU_BRACKET_LL_MARGIN * hwc)
    hi = float(
        np.nanmax(Utr)
        + (float(pden.n_LL) + MU_BRACKET_LL_MARGIN + 1.0) * hwc
    )

    f_lo = nbar(lo) - n_target
    f_hi = nbar(hi) - n_target

    expansion = 0
    while f_lo * f_hi > 0 and expansion < 12:
        lo -= 2.0 * hwc
        hi += 2.0 * hwc
        f_lo = nbar(lo) - n_target
        f_hi = nbar(hi) - n_target
        expansion += 1

    if f_lo * f_hi > 0:
        raise RuntimeError(
            f"Could not bracket mu for frame {index}, B={frame.B_T:.6g} T. "
            f"n(lo)-target={f_lo:.6e}, n(hi)-target={f_hi:.6e}. "
            "Check n_LL, broadening, and density units."
        )

    mu_new = float(
        brentq(
            lambda mu: nbar(mu) - n_target,
            lo,
            hi,
            xtol=1e-10,
            rtol=1e-12,
            maxiter=300,
        )
    )

    n_new = nbar(mu_new)

    result = {
        "index": int(index),
        "B_T": float(frame.B_T),
        "mu_original_meV": mu_old,
        "mu_recomputed_meV": mu_new,
        "delta_mu_meV": mu_new - mu_old,
        "n_target_nm2": float(n_target),
        "target_source": target_source,
        "n_saved_crop_nm2": float(n_saved_crop) if np.isfinite(n_saved_crop) else np.nan,
        "n_at_original_mu_nm2": float(n_old),
        "n_at_recomputed_mu_nm2": float(n_new),
        "relative_density_error_original": float((n_old - n_target) / n_target),
        "relative_density_error_recomputed": float((n_new - n_target) / n_target),
    }

    if verbose:
        print(
            f"B={result['B_T']:.6f} T | "
            f"mu_old={mu_old:.6f} meV | "
            f"mu_new={mu_new:.6f} meV | "
            f"dmu={result['delta_mu_meV']:+.6f} meV"
        )
        print(
            f"target n={n_target:.8e} nm^-2 ({target_source}) | "
            f"n(mu_old)={n_old:.8e} | "
            f"n(mu_new)={n_new:.8e}"
        )

    return result


### Run the density / chemical-potential diagnostic over the whole \(B\) sweep

This is much cheaper than the Kwant sweep because no scattering system is built here.


In [ ]:
mu_selector = hartree_energy_selector(
    key_candidates=[
        "params.mu",
        "mu",
        "chemical_potential",
        "chemical_potential_meV",
        "fermi_energy",
        "fermi_energy_meV",
    ],
    unit="meV",
)

mu_density_diagnostics = [
    recompute_mu_for_frame(i, verbose=False)
    for i in range(len(seq.B_values_T))
]

B_mu = np.array([r["B_T"] for r in mu_density_diagnostics])

mu_original_meV = np.array([
    r["mu_original_meV"] for r in mu_density_diagnostics
])

mu_recomputed_meV = np.array([
    r["mu_recomputed_meV"] for r in mu_density_diagnostics
])

delta_mu_meV = np.array([
    r["delta_mu_meV"] for r in mu_density_diagnostics
])

n_target_crop = np.array([
    r["n_target_nm2"] for r in mu_density_diagnostics
])

n_saved_crop = np.array([
    r["n_saved_crop_nm2"] for r in mu_density_diagnostics
])

n_at_original_mu = np.array([
    r["n_at_original_mu_nm2"] for r in mu_density_diagnostics
])

n_at_recomputed_mu = np.array([
    r["n_at_recomputed_mu_nm2"] for r in mu_density_diagnostics
])

print("Target-density source for first frame:")
print(mu_density_diagnostics[0]["target_source"])

print()
print(
    "max |relative density error| using original mu =",
    np.nanmax(
        np.abs((n_at_original_mu - n_target_crop) / n_target_crop)
    ),
)

print(
    "max |relative density error| using recomputed mu =",
    np.nanmax(
        np.abs((n_at_recomputed_mu - n_target_crop) / n_target_crop)
    ),
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    B_mu,
    mu_original_meV,
    "o-",
    ms=3,
    label="original Hartree mu",
)

ax.plot(
    B_mu,
    mu_recomputed_meV,
    "o-",
    ms=3,
    label="recomputed mu on cropped U",
)

ax.set_xlabel("B [T]")
ax.set_ylabel("chemical potential [meV]")
ax.set_title("Original vs cropped-potential chemical potential")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(B_mu, delta_mu_meV, "o-", ms=3)
ax.axhline(0.0, ls="--", lw=1)

ax.set_xlabel("B [T]")
ax.set_ylabel(r"$\mu_{\rm crop}-\mu_{\rm Hartree}$ [meV]")
ax.set_title("Chemical-potential correction induced by cropping")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    B_mu,
    n_target_crop,
    "-",
    lw=2,
    label="target density in crop",
)

if np.any(np.isfinite(n_saved_crop)):
    ax.plot(
        B_mu,
        n_saved_crop,
        "o",
        ms=3,
        label="saved Hartree density, cropped",
    )

ax.plot(
    B_mu,
    n_at_original_mu,
    "o-",
    ms=3,
    label="cropped U evaluated at original mu",
)

ax.plot(
    B_mu,
    n_at_recomputed_mu,
    "o-",
    ms=3,
    label="cropped U evaluated at recomputed mu",
)

ax.set_xlabel("B [T]")
ax.set_ylabel(r"average $n_e$ [nm$^{-2}$]")
ax.set_title("Density consistency after cropping")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(8, 4))

err_old = (n_at_original_mu - n_target_crop) / n_target_crop
err_new = (n_at_recomputed_mu - n_target_crop) / n_target_crop

ax.plot(
    B_mu,
    100.0 * err_old,
    "o-",
    ms=3,
    label="original mu",
)

ax.plot(
    B_mu,
    100.0 * err_new,
    "o-",
    ms=3,
    label="recomputed mu",
)

ax.axhline(0.0, ls="--", lw=1)

ax.set_xlabel("B [T]")
ax.set_ylabel("density error [%]")
ax.set_title("Error in the cropped-region particle-number constraint")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
USE_RECOMPUTED_MU_FOR_TRANSPORT = True


def transport_mu_for_index(index):
    if USE_RECOMPUTED_MU_FOR_TRANSPORT:
        return float(mu_recomputed_meV[index])
    return float(mu_original_meV[index])


print(
    "Transport will use:",
    "recomputed cropped-potential mu"
    if USE_RECOMPUTED_MU_FOR_TRANSPORT
    else "original Hartree mu",
)


## 3. Visualize full reshaping and the cropped transport region

The left panel shows the full v4 reshaped potential and the two crop boundaries. The right panel is the actual reduced potential passed to Kwant. Thus the regions where the source/drain reshaping occurred are absent from the finite scattering region.


In [ ]:
# Full frame after the ORIGINAL lead-mouth reshaping.
prep2_full = _prepare_frame_full(seq, FRAME_INDEX, cfg2)

xmin_full = float(prep2_full.x_nm.min())
xmax_full = float(prep2_full.x_nm.max())
x_left_crop = xmin_full + source_drain_depth_nm
x_right_crop = xmax_full - source_drain_depth_nm
mu_selector = hartree_energy_selector(
    key_candidates=[
        "params.mu",
        "mu",
        "chemical_potential",
        "chemical_potential_meV",
        "fermi_energy",
        "fermi_energy_meV",
    ],
    unit="meV",
)

frame = seq.frame(FRAME_INDEX)

mu_original_frame_meV = float(mu_selector(frame))
mu_recomputed_frame_meV = float(mu_recomputed_meV[FRAME_INDEX])
mu_meV = transport_mu_for_index(FRAME_INDEX)
omB = 1.73*frame.B_T

print(f"original Hartree mu   = {mu_original_frame_meV:.6f} meV")
print(f"recomputed cropped mu = {mu_recomputed_frame_meV:.6f} meV")
print(f"mu used for transport = {mu_meV:.6f} meV")

# Cropped frame actually used for transport.
prep2 = prepare_frame(seq, FRAME_INDEX, cfg2)

fig, axs = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

im0 = axs[0].pcolormesh(
    prep2_full.x_nm,
    prep2_full.y_nm,
    prep2_full.U_for_transport_meV,
    shading="auto",
    vmin=0,
    vmax=10,
)
axs[0].axvline(x_left_crop, ls="--", lw=2, label="crop boundary")
axs[0].axvline(x_right_crop, ls="--", lw=2)
axs[0].set_aspect("equal")
axs[0].set_xlabel("x [nm]")
axs[0].set_ylabel("y [nm]")
axs[0].set_title("full U after lead-mouth reshaping")
axs[0].legend()
fig.colorbar(im0, ax=axs[0], label="meV")

im1 = axs[1].pcolormesh(
    prep2.x_nm,
    prep2.y_nm,
    prep2.U_for_transport_meV,
    shading="auto",
    vmin=0,
    vmax=10,
)
axs[1].set_aspect("equal")
axs[1].set_xlabel("x [nm]")
axs[1].set_ylabel("y [nm]")
axs[1].set_title("cropped U used by Kwant")
fig.colorbar(im1, ax=axs[1], label="meV")

plt.show()


# Mid-line comparison: the cropped curve is precisely the surviving interior
# of the already-reshaped full transport potential.
plt.figure(figsize=(7, 4))

y_mid_full = len(prep2_full.y_nm) // 2
y_mid_crop = len(prep2.y_nm) // 2

plt.plot(
    prep2_full.x_nm,
    prep2_full.U_for_transport_meV[y_mid_full],
    label="full reshaped U",
)
plt.plot(
    prep2.x_nm,
    prep2.U_for_transport_meV[y_mid_crop],
    lw=2,
    label="cropped U used by Kwant",
)
plt.plot(
    prep2.x_nm,
    prep2.U_for_transport_meV[y_mid_crop] + .5*omB + int(mu_meV/omB)*omB,
    lw=2,
)
plt.plot(
    prep2.x_nm,
    prep2.U_for_transport_meV[y_mid_crop] + .5*omB + int(mu_meV/omB-1)*omB,
    lw=2,
)
plt.plot(
    prep2.x_nm,
    prep2.U_for_transport_meV[y_mid_crop] + .5*omB + int(mu_meV/omB-2)*omB,
    lw=2,
)
plt.axvline(x_left_crop, ls="--", lw=1)
plt.axvline(x_right_crop, ls="--", lw=1)
plt.axhline(mu_meV, ls='--', lw=2)

plt.ylim(0, 15)
plt.xlabel("x [nm]")
plt.ylabel("U(x, y=mid) [meV]")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 4. Build and inspect one reduced two-terminal Kwant system

The builder receives only the cropped `PreparedFrame`, so the finite scattering region contains no sites belonging to the discarded source/drain reshaping sections. The two full-width leads attach to the new left/right boundaries.


In [ ]:
import kwant

builder2 = build_kwant_builder(prep2, cfg2)
fsyst2, phase_params2 = finalize_with_magnetic_field(
    builder2,
    prep2.B_T,
    len(cfg2.lead_specs),
    cfg2.lead_B_mode,
)

print("number of leads:", len(fsyst2.leads))
kwant.plot(fsyst2);

## 5. Run one two-terminal frame

By default the module reads the Hartree chemical potential from metadata. If your metadata uses a different name, adjust `key_candidates`.

In [ ]:
# ============================================================
# ENERGY SWEEP AROUND mu(B) FOR ONE FIXED HARTREE FRAME
# ============================================================

FRAME_INDEX = FRAME_INDEX   # keep your current selected frame

# Energy window around mu.
# Example: +/- 0.05 meV around the Hartree chemical potential.
ENERGY_HALF_WIDTH_MEV = 1.
N_ENERGIES = 61

# Number of parallel jobs.
# n_jobs=-1 uses all available CPU cores.
N_JOBS = -1


# ------------------------------------------------------------
# 1. Get the Hartree chemical potential of this frame
# ------------------------------------------------------------

mu_selector = hartree_energy_selector(
    key_candidates=[
        "params.mu",
        "mu",
        "chemical_potential",
        "chemical_potential_meV",
        "fermi_energy",
        "fermi_energy_meV",
    ],
    unit="meV",
)

frame = seq.frame(FRAME_INDEX)

mu_original_frame_meV = float(mu_selector(frame))
mu_recomputed_frame_meV = float(mu_recomputed_meV[FRAME_INDEX])
mu_meV = transport_mu_for_index(FRAME_INDEX)

energies_meV = np.linspace(
    mu_meV - ENERGY_HALF_WIDTH_MEV,
    mu_meV + ENERGY_HALF_WIDTH_MEV,
    N_ENERGIES,
)

print(f"FRAME_INDEX = {FRAME_INDEX}")
print(f"original Hartree mu   = {mu_original_frame_meV:.6f} meV")
print(f"recomputed cropped mu = {mu_recomputed_frame_meV:.6f} meV")
print(f"mu used for sweep     = {mu_meV:.6f} meV")
print(
    f"Energy sweep: {energies_meV[0]:.6f} -> "
    f"{energies_meV[-1]:.6f} meV"
)
print(f"N_ENERGIES = {N_ENERGIES}")


In [ ]:
# ------------------------------------------------------------
# 2. Helper: constant-energy selector
# ------------------------------------------------------------
#
# run_one_frame expects a selector acting on the Hartree frame.
# For the energy sweep we simply ignore the frame and return
# the desired scattering energy.
#

def constant_energy_selector(E_meV):
    def selector(_frame):
        return float(E_meV)
    return selector


# ------------------------------------------------------------
# 3. One Kwant calculation at one energy
# ------------------------------------------------------------

def run_at_energy(E_meV):
    selector_E = constant_energy_selector(E_meV)

    prep, smat, T = run_one_frame(
        seq,
        index=FRAME_INDEX,
        cfg=cfg2,
        energy_selector=selector_E,
    )

    G = two_terminal_conductance_e2h(
        T,
        source=0,
        drain=1,
    )


    return {
        "E_meV": float(E_meV),
        "B_T": float(prep.B_T),
        "G_e2h": float(G),
        "T_matrix": np.asarray(T),
    }


# ------------------------------------------------------------
# 4. Parallel energy sweep
# ------------------------------------------------------------

results_E = Parallel(
    n_jobs=N_JOBS,
    verbose=10,
)(
    delayed(run_at_energy)(E)
    for E in energies_meV
)


# ------------------------------------------------------------
# 5. Collect results
# ------------------------------------------------------------

E_arr = np.array([r["E_meV"] for r in results_E])
G_arr = np.array([r["G_e2h"] for r in results_E])

B_T = results_E[0]["B_T"]

print()
print(f"B = {B_T:.6f} T")
print(f"mu = {mu_meV:.6f} meV")
print(f"G(mu) ~ {G_arr[np.argmin(np.abs(E_arr - mu_meV))]:.6f} e^2/h")



In [ ]:
# ------------------------------------------------------------
# 6. Plot G(E)
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(E_arr, G_arr, "o-", ms=3)
ax.axvline(
    mu_meV,
    ls="--",
    lw=1.5,
    label=r"$\mu(B)$",
)

ax.set_xlabel("Scattering energy E [meV]")
ax.set_ylabel(r"$G_{2t}$ [$e^2/h$]")
ax.set_title(
    rf"Energy-dependent transmission, $B={B_T:.3f}$ T"
)

ax.grid(alpha=0.3)
ax.legend()
ax.set_ylim(0,10)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 7. Optional: plot versus E - mu
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(E_arr - mu_meV, G_arr, "o-", ms=3)
ax.axvline(0.0, ls="--", lw=1.5)

ax.set_xlabel(r"$E-\mu$ [meV]")
ax.set_ylabel(r"$G_{2t}$ [$e^2/h$]")
ax.set_title(
    rf"Energy sweep around $\mu$, $B={B_T:.3f}$ T"
)

ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
wavefunctions = []

for i, energy in enumerate(energies_meV):

    print(
        f"\rComputing {i+1}/{len(energies_meV)} "
        f"E = {energy:.6f} meV",
        end=""
    )

    wfs = kwant.wave_function(
        fsyst2,
        energy=float(energy),
        params=phase_params2,
    )

    # All scattering states incoming from lead 0
    scattering_wf = wfs(0)

    wavefunctions.append(scattering_wf)

print("\nDone.")

In [ ]:
# Choose what to visualize
SUM_ALL_MODES = True
MODE_INDEX = -1


fig, ax = plt.subplots(dpi=120, figsize=(8, 4))


def init():
    ax.clear()
    ax.set_title("Wavefunction Animation")


def update(frame):

    ax.clear()

    psi = wavefunctions[frame]
    energy = energies_meV[frame]

    if SUM_ALL_MODES:
        density = np.sum(np.abs(psi)**2, axis=0)
        mode_text = "all incoming modes"
    else:
        density = np.abs(psi[MODE_INDEX])**2
        mode_text = f"mode {MODE_INDEX}"

    kwant.plotter.map(
        fsyst2,
        density,
        ax=ax,
        show=False,
    )

    ax.set_title(
        rf"$B={prep2.B_T:.3f}$ T, "
        rf"$E-\mu={energy-mu_meV:+.4f}$ meV"
        "\n"
        + mode_text
    )


ani = FuncAnimation(
    fig,
    update,
    frames=len(energies_meV),
    init_func=init,
    repeat=False,
    interval=100,
)

plt.close(fig)

HTML(ani.to_jshtml())